In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# smart_hybrid_geocoder.py

import re
from geopy.geocoders import Nominatim
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# -----------------------------
# 1. Setup
# -----------------------------
geolocator = Nominatim(user_agent="smart_geo_pipeline")

model = SentenceTransformer("all-MiniLM-L6-v2")

KNOWN_REGIONS = [
    "Westlands Nairobi Kenya",
    "Kilimani Nairobi Kenya",
    "Nairobi CBD Kenya",
    "Karen Nairobi Kenya",
    "Mombasa Kenya"
]

region_embeddings = model.encode(KNOWN_REGIONS)


# -----------------------------
# 2. Normalize Text
# -----------------------------
def normalize_text(address: str) -> str:
    address = address.lower()
    address = re.sub(r"[^\w\s]", "", address)
    address = re.sub(r"\s+", " ", address).strip()
    return address


# -----------------------------
# 3. Embedding Similarity Check
# -----------------------------
def get_best_match(user_input: str):
    input_embedding = model.encode([user_input])
    similarities = cosine_similarity(input_embedding, region_embeddings)

    best_idx = np.argmax(similarities)
    best_score = similarities[0][best_idx]

    return KNOWN_REGIONS[best_idx], best_score


# -----------------------------
# 4. Nominatim Geocoding
# -----------------------------
def geocode(address: str):
    try:
        location = geolocator.geocode(address, timeout=5)
        if location:
            return {
                "input": address,
                "resolved_address": location.address,
                "latitude": round(location.latitude, 5),
                "longitude": round(location.longitude, 5)
            }
    except Exception as e:
        print(f"Nominatim error: {e}")
    return None


# -----------------------------
# 5. Smart Hybrid Pipeline
# -----------------------------
def smart_geocode(raw_address: str, threshold: float = 0.75):

    print(f"\n🔍 Input: {raw_address}")

    # Step 1: Normalize
    clean = normalize_text(raw_address)

    # Step 2: Embedding similarity
    best_match, score = get_best_match(clean)

    print(f"🧠 Best embedding match: {best_match} (score={score:.2f})")

    # Step 3: Decision
    if score >= threshold:
        print("✅ High similarity → using matched region")
        query = best_match
        source = "embedding"
    else:
        print("⚠️ Low similarity → using raw input")
        query = clean
        source = "raw"

    # Step 4: Geocode
    result = geocode(query)

    if result:
        result["strategy"] = source
        result["similarity_score"] = float(score)
        return result

    return None


# -----------------------------
# 6. Test Cases
# -----------------------------
if __name__ == "__main__":

    test_inputs = [
        "westlnds nrb",       # messy → should use embedding
        "kilimani nairobi",   # clean → embedding ok
        "unknown place xyz",  # low similarity → raw
        "mombasa kenya"
    ]

    for addr in test_inputs:
        result = smart_geocode(addr)
        print("📍 Result:", result)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


🔍 Input: westlnds nrb
🧠 Best embedding match: Westlands Nairobi Kenya (score=0.32)
⚠️ Low similarity → using raw input
📍 Result: None

🔍 Input: kilimani nairobi
🧠 Best embedding match: Kilimani Nairobi Kenya (score=0.98)
✅ High similarity → using matched region
📍 Result: {'input': 'Kilimani Nairobi Kenya', 'resolved_address': 'Kilimani, Kilimani division, Westlands, Nairobi, 44847, Kenya', 'latitude': -1.29601, 'longitude': 36.78178, 'strategy': 'embedding', 'similarity_score': 0.9777976274490356}

🔍 Input: unknown place xyz
🧠 Best embedding match: Westlands Nairobi Kenya (score=0.22)
⚠️ Low similarity → using raw input
📍 Result: None

🔍 Input: mombasa kenya
🧠 Best embedding match: Mombasa Kenya (score=1.00)
✅ High similarity → using matched region
📍 Result: {'input': 'Mombasa Kenya', 'resolved_address': 'Mombasa, Tononoka ward, Mvita, Mombasa, 80100, Kenya', 'latitude': -4.05052, 'longitude': 39.66717, 'strategy': 'embedding', 'similarity_score': 1.000000238418579}
